# Assault DDQN - HU009 full training orchestration


## 1. Bootstrap Local -> GitHub -> Colab

In [1]:
import os

os.environ["ASSAULT_BOOTSTRAP_REF"] = "main"
os.environ.pop("ASSAULT_BOOTSTRAP_COMMIT", None)

os.environ["ASSAULT_TRAINING_PROFILE"] = "full"
os.environ["ASSAULT_PROJECT_RUN_ID"] = "assault_ddqn_full_001"
os.environ["ASSAULT_TARGET_TIMESTEPS"] = "250000"
os.environ["ASSAULT_REQUESTED_MODE"] = "auto"

os.environ["ASSAULT_EVALUATION_EPISODES"] = "10"
os.environ["ASSAULT_EVALUATION_EPSILON"] = "0.0"


from pathlib import Path
import os


try:
    from google.colab import drive  # type: ignore
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/reinforcement_learning_reto_1")
else:
    BASE = Path.cwd()

os.environ.setdefault("ASSAULT_BOOTSTRAP_REF", "main")
os.environ.setdefault("ASSAULT_MLFLOW_TRACKING_URI", (BASE / "mlruns").as_uri())
os.environ.setdefault("ASSAULT_CHECKPOINT_DIR", str(BASE / "checkpoints"))
os.environ.setdefault("ASSAULT_TENSORBOARD_DIR", str(BASE / "tensorboard"))

print("Persistent storage configured")
print("BASE:", BASE)


Mounted at /content/drive
Persistent storage configured
BASE: /content/drive/MyDrive/reinforcement_learning_reto_1


In [2]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/j-mauro-r/reinforcement_learning_reto_1.git"
COLAB_ROOT = Path("/content/reinforcement_learning_reto_1")
BOOTSTRAP_REF = os.environ.get("ASSAULT_BOOTSTRAP_REF", "main")
BOOTSTRAP_COMMIT = os.environ.get("ASSAULT_BOOTSTRAP_COMMIT") or None
INSTALL_DEPENDENCIES = os.environ.get("ASSAULT_INSTALL_DEPENDENCIES", "1") == "1"


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


def _git_output(args, cwd):
    return subprocess.check_output(["git", *args], cwd=str(cwd), text=True).strip()


if _running_in_colab():
    if not (COLAB_ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=str(COLAB_ROOT), check=True)
    provisional_ref = BOOTSTRAP_COMMIT or f"origin/{BOOTSTRAP_REF}"
    provisional_sha = _git_output(["rev-parse", "--verify", f"{provisional_ref}^{{commit}}"], COLAB_ROOT)
    subprocess.run(["git", "checkout", "--detach", provisional_sha], cwd=str(COLAB_ROOT), check=True)
    ASSAULT_DIR = COLAB_ROOT / "2_Assault"
else:
    PROJECT_ROOT = Path(_git_output(["rev-parse", "--show-toplevel"], Path.cwd()))
    ASSAULT_DIR = PROJECT_ROOT / "2_Assault"

for path in (ASSAULT_DIR, ASSAULT_DIR.parent):
    value = str(path.resolve())
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from src.execution_bootstrap import (
    install_project_requirements,
    prepare_execution_environment,
    verify_environment_import,
)

bootstrap = prepare_execution_environment(
    requested_ref=BOOTSTRAP_REF,
    requested_commit=BOOTSTRAP_COMMIT,
    repo_url=REPO_URL,
    colab_root=COLAB_ROOT,
)

PROJECT_ROOT = bootstrap.repo_root
ASSAULT_DIR = bootstrap.assault_dir

if INSTALL_DEPENDENCIES:
    install_project_requirements(bootstrap.requirements_path)

environment_source = verify_environment_import(bootstrap)
bootstrap.as_dict()


Execution bootstrap
  runtime: Google Colab
  repository: /content/reinforcement_learning_reto_1
  assault_dir: /content/reinforcement_learning_reto_1/2_Assault
  requested_ref: main
  requested_commit: <none>
  resolved_sha: deba2e709addcf3f09b2ab089db652f07abec8bf
  requirements: /content/reinforcement_learning_reto_1/2_Assault/requirements.txt
src.environment import source: /content/reinforcement_learning_reto_1/2_Assault/src/environment.py


{'is_colab': True,
 'repo_root': '/content/reinforcement_learning_reto_1',
 'assault_dir': '/content/reinforcement_learning_reto_1/2_Assault',
 'requested_ref': 'main',
 'requested_commit': None,
 'resolved_sha': 'deba2e709addcf3f09b2ab089db652f07abec8bf',
 'requirements_path': '/content/reinforcement_learning_reto_1/2_Assault/requirements.txt',
 'environment_source': '/content/reinforcement_learning_reto_1/2_Assault/src/environment.py'}

In [3]:
# Optional notebook diagnostics go after the project bootstrap/import cells.
# This placeholder intentionally avoids importing src before sys.path is configured.


## 2. Imports, configuration and tracking


In [4]:
from pathlib import Path

from src.callbacks import TensorBoardLogger, load_tensorboard_scalars
from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.evaluator import evaluate_agent
from src.preflight import run_preflight_checks
from src.session_bootstrap import (
    inspect_experiment_state,
    prepare_training_session,
    update_experiment_state_after_success,
)
from src.tracking import MLflowTracker
from src.training_profiles import (
    assert_training_can_start,
    FULL_TRAINING_TARGET_TIMESTEPS,
    evaluate_full_training_ready,
    resolve_training_profile,
)
from src.training_session import run_training_session
from src.utils import get_runtime_info, load_yaml_config

BASE_CONFIG = load_yaml_config(ASSAULT_DIR / "configs" / "ddqn_config.yaml")
TRAINING_PROFILE = os.environ.get("ASSAULT_TRAINING_PROFILE", "smoke").strip().lower()
_TARGET_OVERRIDE = os.environ.get("ASSAULT_TARGET_TIMESTEPS")
profile_context = resolve_training_profile(
    BASE_CONFIG,
    TRAINING_PROFILE,
    target_timesteps=int(_TARGET_OVERRIDE) if _TARGET_OVERRIDE else None,
)
config = profile_context.config
seed = int(config["reproducibility"]["seed"])
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ASSAULT_DIR:", ASSAULT_DIR)
print("BOOTSTRAP_REF:", BOOTSTRAP_REF)
print("BOOTSTRAP_COMMIT:", BOOTSTRAP_COMMIT or "<none>")
print("EXECUTED_SHA:", bootstrap.resolved_sha)
print("src.environment:", environment_source)
print("TRAINING_PROFILE=", profile_context.name)
print("PROFILE_TARGET_TIMESTEPS=", profile_context.target_timesteps)
print("FULL_TRAINING_TARGET_TIMESTEPS=", FULL_TRAINING_TARGET_TIMESTEPS)
print("REPLAY_BUFFER_CAPACITY=", config["replay_buffer"]["capacity"])
print("REPLAY_BUFFER_ESTIMATED_GIB=", round(profile_context.replay_buffer_memory.total_gib, 3))
config


PROJECT_ROOT: /content/reinforcement_learning_reto_1
ASSAULT_DIR: /content/reinforcement_learning_reto_1/2_Assault
BOOTSTRAP_REF: main
BOOTSTRAP_COMMIT: <none>
EXECUTED_SHA: deba2e709addcf3f09b2ab089db652f07abec8bf
src.environment: /content/reinforcement_learning_reto_1/2_Assault/src/environment.py
TRAINING_PROFILE= full
PROFILE_TARGET_TIMESTEPS= 250000
FULL_TRAINING_TARGET_TIMESTEPS= 250000
REPLAY_BUFFER_CAPACITY= 50000
REPLAY_BUFFER_ESTIMATED_GIB= 2.629


{'environment': {'id': 'ALE/Assault-v5',
  'obs_type': 'rgb',
  'frame_skip': 4,
  'repeat_action_probability': 0.25,
  'full_action_space': False,
  'render_mode': None},
 'preprocessing': {'grayscale': True,
  'resize_height': 84,
  'resize_width': 84,
  'frame_stack': 4,
  'dtype': 'uint8',
  'normalize_pixels_in_env': False},
 'reproducibility': {'seed': 42},
 'evaluation': {'episodes': 5, 'epsilon': 0.0, 'max_steps_per_episode': None},
 'network': {'input_channels': 4, 'num_actions': 7},
 'agent': {'gamma': 0.99,
  'learning_rate': 0.0001,
  'epsilon_start': 1.0,
  'epsilon_final': 0.01},
 'replay_buffer': {'capacity': 50000, 'batch_size': 32},
 'training': {'total_timesteps': 250000,
  'learning_starts': 10000,
  'train_frequency': 4,
  'target_update_frequency': 1000,
  'epsilon_decay_steps': 200000},
 'checkpointing': {'enabled': True,
  'interval_steps': 25000,
  'directory': 'checkpoints',
  'mode': 'new',
  'run_id': 'assault_ddqn_exp_001',
  'resume_checkpoint': None,
  'sa

## 3. Runtime and hardware

In [5]:
runtime_info = get_runtime_info()
runtime_info


{'python_version': '3.13.15',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
 'gymnasium_version': '1.1.1',
 'ale_py_version': '0.10.1',
 'cpu': 'x86_64',
 'cpu_count_logical': 12,
 'cpu_count_physical': 6,
 'ram_total_gb': 83.47,
 'ram_available_gb': 81.25,
 'gpu_available': True,
 'gpu_name': 'NVIDIA A100-SXM4-40GB',
 'gpu_vram_total_gb': 39.49,
 'cuda_version': '12.8',
 'torch_version': '2.11.0+cu128'}

## 4. HU002 environment contract

In [6]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_env = create_assault_env(config, mode="eval", seed=seed + 1)

obs, info = train_env.reset(seed=seed)
metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)

print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Action space:", train_env.action_space)
print("Action meanings:", train_env.unwrapped.get_action_meanings())
print("Initial info:", info)
print("Metadata:", metadata)


Observation shape: (4, 84, 84)
Observation dtype: uint8
Action space: Discrete(7)
Action meanings: ['NOOP', 'FIRE', 'UP', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']
Initial info: {'lives': 4, 'episode_frame_number': 0, 'frame_number': 0, 'seeds': (np.uint32(3444837047), np.uint32(2669555309))}
Metadata: EnvironmentMetadata(env_id='ALE/Assault-v5', mode='train', seed=42, action_space='Discrete(7)', action_meanings=('NOOP', 'FIRE', 'UP', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE'), observation_shape=(4, 84, 84), observation_dtype='uint8', base_frameskip=4, wrapper_frameskip=1, effective_frameskip=4, repeat_action_probability=0.25, full_action_space=False)


## 5. HU002 autovalidations

In [7]:
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert train_env.observation_space.shape == eval_env.observation_space.shape
assert train_env.observation_space.dtype == eval_env.observation_space.dtype
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)

obs, info = train_env.reset(seed=seed)
for step in range(100):
    action = int(train_env.action_space.sample())
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert obs.shape == (4, 84, 84)
    assert str(obs.dtype) == "uint8"
    if terminated or truncated:
        obs, info = train_env.reset()

print("HU002 validations passed.")
train_env.close()
eval_env.close()


HU002 validations passed.


## 6. HU004 preflight gate

In [8]:
preflight_report = run_preflight_checks(config)
print(preflight_report.format_summary())
preflight_report.as_dict()


===== DDQN PRE-FLIGHT =====
Runtime: Google Colab
Device: cuda
Device: PASS
Environment: PASS
Observation: PASS (4, 84, 84) uint8
QNetwork: PASS -> (1, 7)
ReplayBuffer: PASS
DDQN update: PASS loss=0.000926
Loss finite: PASS
Target stable: PASS
Target sync: PASS
Save/load: PASS temporary_file_cleaned=True

READY_FOR_TRAINING=True


{'passed': True,
 'ready_for_training': True,
 'runtime': 'Google Colab',
 'device': 'cuda',
 'checks': {'Device': True,
  'Environment': True,
  'Observation': True,
  'QNetwork': True,
  'ReplayBuffer': True,
  'DDQN update': True,
  'Loss finite': True,
  'Target stable': True,
  'Target sync': True,
  'Save/load': True},
 'errors': [],
 'details': {'runtime_info': {'python_version': '3.13.15',
   'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
   'gymnasium_version': '1.1.1',
   'ale_py_version': '0.10.1',
   'cpu': 'x86_64',
   'cpu_count_logical': 12,
   'cpu_count_physical': 6,
   'ram_total_gb': 83.47,
   'ram_available_gb': 81.19,
   'gpu_available': True,
   'gpu_name': 'NVIDIA A100-SXM4-40GB',
   'gpu_vram_total_gb': 39.49,
   'cuda_version': '12.8',
   'torch_version': '2.11.0+cu128'},
  'Observation': '(4, 84, 84) uint8',
  'QNetwork': '-> (1, 7)',
  'DDQN update': 'loss=0.000926',
  'Save/load': 'temporary_file_cleaned=True'}}

## 7. Abort if preflight fails

In [9]:
if not preflight_report.ready_for_training:
    raise RuntimeError("READY_FOR_TRAINING=False; HU005 training aborted.")
print("READY_FOR_TRAINING=True")


READY_FOR_TRAINING=True


## 8. HU009 profile and automated session bootstrap


In [10]:
def _optional_env(name: str, default=None):
    value = os.environ.get(name)
    if value is None or not value.strip():
        return default
    return value.strip()



mlflow_config = config.get("mlflow", {})
evaluation_config = config.get("evaluation", {})
PROJECT_RUN_ID = _optional_env("ASSAULT_PROJECT_RUN_ID", config["checkpointing"]["run_id"] + f"_{profile_context.name}_hu009")
REQUESTED_MODE = _optional_env("ASSAULT_REQUESTED_MODE", "auto").lower()
TARGET_TIMESTEPS = profile_context.target_timesteps
RESUME_MODE = _optional_env("ASSAULT_RESUME_MODE", "resume_full")
EVALUATION_EPISODES = int(_optional_env("ASSAULT_EVALUATION_EPISODES", evaluation_config.get("episodes", 2)))
EVALUATION_EPSILON = float(_optional_env("ASSAULT_EVALUATION_EPSILON", evaluation_config.get("epsilon", 0.0)))
_evaluation_max_steps = _optional_env("ASSAULT_EVALUATION_MAX_STEPS", evaluation_config.get("max_steps_per_episode"))
EVALUATION_MAX_STEPS = int(_evaluation_max_steps) if _evaluation_max_steps is not None else None

session_context = prepare_training_session(
    base_path=BASE,
    project_run_id=PROJECT_RUN_ID,
    target_timesteps=TARGET_TIMESTEPS,
    requested_mode=REQUESTED_MODE,
    config=config,
    checkpoint_root=os.environ.get("ASSAULT_CHECKPOINT_DIR"),
    tensorboard_root=os.environ.get("ASSAULT_TENSORBOARD_DIR"),
    tracking_uri=os.environ.get("ASSAULT_MLFLOW_TRACKING_URI"),
    resume_mode=RESUME_MODE,
    bootstrap_ref=BOOTSTRAP_REF,
    bootstrap_commit=bootstrap.resolved_sha,
)

selected_runtime = "Google Colab" if _running_in_colab() else "local"
selected_device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
full_training_gate = evaluate_full_training_ready(
    profile_context=profile_context,
    session_context=session_context,
    preflight_report=preflight_report,
    runtime_info=runtime_info,
    observation_shape=tuple(obs.shape),
    observation_dtype=str(obs.dtype),
    action_space=str(metadata.action_space),
    runtime=selected_runtime,
    device=selected_device,
)
FULL_TRAINING_READY = full_training_gate.ready

RUN_ID = session_context.project_run_id
CHECKPOINT_DIR = session_context.checkpoint_root
TENSORBOARD_DIR = session_context.tensorboard_root
MLFLOW_TRACKING_MODE = session_context.tracking_mode
MLFLOW_RUN_ID = session_context.mlflow_run_id
MLFLOW_SESSION_ID = session_context.tracking_session_id
CHECKPOINT_INPUT_REFERENCE = str(session_context.checkpoint_input) if session_context.checkpoint_input else None
SESSION_TARGET_TIMESTEPS = session_context.target_timesteps

print("TRAINING_PROFILE=", profile_context.name)
print("SESSION_BOOTSTRAP_READY=True")
print("FULL_TRAINING_READY=", FULL_TRAINING_READY)
assert_training_can_start(profile_context, full_training_gate)
print("project_run_id=", session_context.project_run_id)
print("tracking_mode=", session_context.tracking_mode)
print("mlflow_run_id=", session_context.mlflow_run_id or "<new run>")
print("tracking_session_id=", session_context.tracking_session_id)
print("checkpoint_input=", session_context.checkpoint_input)
print("restored_expected_step=", session_context.restored_expected_step)
print("target_global_step=", session_context.target_timesteps)
print("bootstrap_commit=", session_context.bootstrap_commit)
print("config_fingerprint=", session_context.config_fingerprint)
print("replay_buffer_capacity=", profile_context.replay_buffer_memory.capacity)
print("replay_buffer_estimated_memory_gib=", round(profile_context.replay_buffer_memory.total_gib, 3))
print("runtime_ram_available_gib=", full_training_gate.ram_available_gib)
print("runtime_ram_margin_gib=", full_training_gate.ram_margin_gib)
print("device=", selected_device)
print("checkpoint_root=", session_context.checkpoint_root)
print("tensorboard_root=", session_context.tensorboard_root)
print("tracking_uri=", session_context.tracking_uri)
print("full_training_gate_issues=", full_training_gate.issues)


TRAINING_PROFILE= full
SESSION_BOOTSTRAP_READY=True
FULL_TRAINING_READY= True
project_run_id= assault_ddqn_full_001
tracking_mode= new
mlflow_run_id= <new run>
tracking_session_id= session_001
checkpoint_input= None
restored_expected_step= None
target_global_step= 250000
bootstrap_commit= deba2e709addcf3f09b2ab089db652f07abec8bf
config_fingerprint= 936da39d3090bb18f6103631a7fcbbae794c13fcb4418c6c2766d96b1409776e
replay_buffer_capacity= 50000
replay_buffer_estimated_memory_gib= 2.629
runtime_ram_available_gib= 81.25
runtime_ram_margin_gib= 78.62082980573177
device= cuda
checkpoint_root= /content/drive/MyDrive/reinforcement_learning_reto_1/checkpoints
tensorboard_root= /content/drive/MyDrive/reinforcement_learning_reto_1/tensorboard
tracking_uri= file:///content/drive/MyDrive/reinforcement_learning_reto_1/mlruns
full_training_gate_issues= []


## 9. HU009 tracked training session


In [11]:
import copy

MLFLOW_TRACKING_PASS = False
MLFLOW_SESSION_ARTIFACTS = []
CHECKPOINT_INPUT_LOADED = False
RESTORED_GLOBAL_STEP = None
REPLAY_BUFFER_RESTORED = False
MULTISESSION_CHECKPOINT_RESUME_PASS = False
EXPERIMENT_MANIFEST_UPDATED = False

session_config = copy.deepcopy(config)
session_config.setdefault("training", {})["total_timesteps"] = SESSION_TARGET_TIMESTEPS
session_config.setdefault("mlflow", {})["tracking_uri"] = session_context.tracking_uri
session_config.setdefault("mlflow", {})["tracking_mode"] = session_context.tracking_mode
session_config.setdefault("mlflow", {})["mlflow_run_id"] = session_context.mlflow_run_id
session_config.setdefault("mlflow", {})["tracking_session_id"] = session_context.tracking_session_id

mlflow_tracker = MLflowTracker.from_config(session_config)
mlflow_metadata = mlflow_tracker.start_run(
    project_run_id=session_context.project_run_id,
    tracking_mode=session_context.tracking_mode,
    mlflow_run_id=session_context.mlflow_run_id,
    run_name=session_context.project_run_id,
    tags={"stage": "HU008B"},
    tracking_session_id=session_context.tracking_session_id,
)
print("MLflow tracking URI:", mlflow_metadata.tracking_uri)
print("MLflow experiment name:", mlflow_metadata.experiment_name)
print("project_run_id:", mlflow_metadata.project_run_id)
print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)
print("tracking_session_id:", mlflow_metadata.tracking_session_id)

try:
    mlflow_tracker.log_run_context(
        config=session_config,
        runtime_info=runtime_info,
        git_commit=session_context.bootstrap_commit,
        git_ref=session_context.bootstrap_ref,
        project_run_id=session_context.project_run_id,
        action_space=str(metadata.action_space),
        observation_dtype=str(obs.dtype),
        runtime=selected_runtime,
        device=selected_device,
    )
    mlflow_tracker.log_config_snapshot(config, artifact_file="config/base_config.json")
    effective_config_artifact = mlflow_tracker.log_session_config(
        session_config,
        tracking_session_id=session_context.tracking_session_id,
    )
    mlflow_tracker.log_runtime_metadata(
        runtime_info=runtime_info,
        git_commit=session_context.bootstrap_commit,
        runtime=selected_runtime,
        tracking_session_id=session_context.tracking_session_id,
    )

    session_summary = run_training_session(
        config=session_config,
        checkpoint_root=session_context.checkpoint_root,
        tensorboard_root=session_context.tensorboard_root,
        run_id=session_context.project_run_id,
        repo_path=PROJECT_ROOT,
        tracking_mode=session_context.tracking_mode,
        checkpoint_input=session_context.checkpoint_input,
        resume_mode=session_context.resume_mode,
        total_timesteps=session_context.target_timesteps,
        device=selected_device,
    )
    session_result = session_summary.as_dict()
    checkpoint_output_reference = session_summary.checkpoint_output_reference
    session_initial_step = int(session_summary.initial_global_step)
    session_final_step = int(session_summary.final_global_step)
    CHECKPOINT_INPUT_LOADED = bool(session_summary.checkpoint_input_loaded)
    RESTORED_GLOBAL_STEP = session_summary.restored_global_step
    REPLAY_BUFFER_RESTORED = bool(session_summary.replay_buffer_restored)
    MULTISESSION_CHECKPOINT_RESUME_PASS = bool(
        session_context.tracking_mode == "resume"
        and CHECKPOINT_INPUT_LOADED
        and RESTORED_GLOBAL_STEP == session_initial_step
        and session_context.restored_expected_step == session_initial_step
        and REPLAY_BUFFER_RESTORED
        and session_final_step > session_initial_step
        and session_summary.checkpoint_input_reference == CHECKPOINT_INPUT_REFERENCE
    )

    eval_env = create_assault_env(session_config, mode="eval", seed=seed + 10_000)
    try:
        evaluation_summary = evaluate_agent(
            eval_env,
            session_summary.agent,
            episodes=EVALUATION_EPISODES,
            epsilon=EVALUATION_EPSILON,
            max_steps_per_episode=EVALUATION_MAX_STEPS,
        )
    finally:
        eval_env.close()

    session_result = session_summary.as_dict()
    session_result["evaluation"] = evaluation_summary.as_dict()
    session_result["session_bootstrap"] = session_context.as_dict()
    session_result["training_profile"] = profile_context.as_dict()
    session_result["full_training_gate"] = full_training_gate.as_dict()

    mlflow_tracker.log_training_summary(session_summary.training, tracking_session_id=session_context.tracking_session_id)
    mlflow_tracker.log_evaluation_summary(evaluation_summary, tracking_session_id=session_context.tracking_session_id)
    mlflow_tracker.log_checkpoint_reference(
        session_summary.checkpoint,
        resume_mode=session_context.resume_mode if session_context.tracking_mode == "resume" else "new",
        project_run_id=session_context.project_run_id,
        checkpoint_input_reference=CHECKPOINT_INPUT_REFERENCE,
        checkpoint_output_reference=checkpoint_output_reference,
        tracking_session_id=session_context.tracking_session_id,
    )
    mlflow_tracker.log_dict_artifact(
        session_result,
        "training_session_summary.json",
        tracking_session_id=session_context.tracking_session_id,
    )
    mlflow_tracker.log_session_metadata(
        tracking_mode=session_context.tracking_mode,
        runtime_info=runtime_info,
        git_commit=session_context.bootstrap_commit,
        git_ref=session_context.bootstrap_ref,
        runtime=selected_runtime,
        device=selected_device,
        initial_global_step=session_initial_step,
        final_global_step=session_final_step,
        session_target_timesteps=session_context.target_timesteps,
        checkpoint_input_reference=CHECKPOINT_INPUT_REFERENCE,
        checkpoint_output_reference=checkpoint_output_reference,
        checkpoint_input_loaded=CHECKPOINT_INPUT_LOADED,
        restored_checkpoint_path=session_summary.restored_checkpoint_path,
        restored_global_step=RESTORED_GLOBAL_STEP,
        replay_buffer_restored=REPLAY_BUFFER_RESTORED,
        resume_mode=session_summary.resume_mode,
        effective_config_artifact=effective_config_artifact,
        tracking_session_id=session_context.tracking_session_id,
        extra={
            "config_fingerprint": session_context.config_fingerprint,
            "training_profile": profile_context.name,
            "replay_buffer_memory": profile_context.replay_buffer_memory.as_dict(),
            "full_training_ready": FULL_TRAINING_READY,
        },
    )
    MLFLOW_SESSION_ARTIFACTS = [
        artifact.path for artifact in mlflow_tracker.list_session_artifacts(session_context.tracking_session_id)
    ]
    queried_run = mlflow_tracker.get_run(mlflow_metadata.mlflow_run_id) if mlflow_metadata.enabled else None
    required_session_artifacts = {
        f"sessions/{session_context.tracking_session_id}/session_metadata.json",
        f"sessions/{session_context.tracking_session_id}/runtime.json",
        f"sessions/{session_context.tracking_session_id}/training_summary.json",
        f"sessions/{session_context.tracking_session_id}/evaluation_summary.json",
        f"sessions/{session_context.tracking_session_id}/effective_config.json",
        f"sessions/{session_context.tracking_session_id}/checkpoint_reference.json",
        f"sessions/{session_context.tracking_session_id}/training_session_summary.json",
    }
    MLFLOW_TRACKING_PASS = bool(
        not mlflow_metadata.enabled
        or (
            queried_run is not None
            and queried_run.info.run_id == mlflow_metadata.mlflow_run_id
            and queried_run.data.params.get("identity.project_run_id") == session_context.project_run_id
            and queried_run.data.tags.get("latest_tracking_session_id") == session_context.tracking_session_id
            and "train/final_global_step" in queried_run.data.metrics
            and "eval/mean_reward" in queried_run.data.metrics
            and required_session_artifacts.issubset(set(MLFLOW_SESSION_ARTIFACTS))
            and (session_context.tracking_mode != "resume" or MULTISESSION_CHECKPOINT_RESUME_PASS)
        )
    )
except Exception:
    mlflow_tracker.end_run(status="FAILED")
    raise
else:
    mlflow_tracker.end_run(status="FINISHED")
    experiment_state = update_experiment_state_after_success(
        session_context,
        mlflow_metadata.mlflow_run_id,
        checkpoint_output_reference,
        session_final_step,
    )
    EXPERIMENT_MANIFEST_UPDATED = True

session_result


MLflow tracking URI: file:///content/drive/MyDrive/reinforcement_learning_reto_1/mlruns
MLflow experiment name: assault_ddqn
project_run_id: assault_ddqn_full_001
mlflow_run_id: 048a4acf1a3541b1bcfc88b278070912
tracking_session_id: session_001


{'tracking_mode': 'new',
 'run_id': 'assault_ddqn_full_001',
 'initial_global_step': 0,
 'final_global_step': 250000,
 'checkpoint_input_reference': None,
 'checkpoint_input_loaded': False,
 'restored_checkpoint_path': None,
 'restored_global_step': None,
 'replay_buffer_restored': False,
 'resume_mode': None,
 'checkpoint_output_reference': '/content/drive/MyDrive/reinforcement_learning_reto_1/checkpoints/assault_ddqn_full_001/checkpoint_step_250000.pt',
 'device': 'cuda',
 'checkpoint': {'path': '/content/drive/MyDrive/reinforcement_learning_reto_1/checkpoints/assault_ddqn_full_001/checkpoint_step_250000.pt',
  'run_id': 'assault_ddqn_full_001',
  'checkpoint_step': 250000,
  'size_bytes': 2881543669,
  'save_replay_buffer': True},
 'training': {'global_step': 250000,
  'episodes_completed': 417,
  'episode_rewards': [189.0,
   168.0,
   168.0,
   147.0,
   105.0,
   105.0,
   378.0,
   168.0,
   294.0,
   168.0,
   210.0,
   210.0,
   210.0,
   189.0,
   105.0,
   168.0,
   210.0,
 

## 10. HU009 result gates


In [12]:
assert session_summary.initial_global_step == session_initial_step
assert session_summary.final_global_step == session_final_step
assert session_final_step == session_context.target_timesteps
assert session_summary.checkpoint.path.exists()
assert evaluation_summary.episodes == EVALUATION_EPISODES
assert evaluation_summary.epsilon == EVALUATION_EPSILON
assert EXPERIMENT_MANIFEST_UPDATED
if session_context.tracking_mode == "new":
    assert session_initial_step == 0
    assert not CHECKPOINT_INPUT_LOADED
    assert CHECKPOINT_INPUT_REFERENCE is None
    assert session_context.restored_expected_step is None
elif session_context.tracking_mode == "resume":
    assert CHECKPOINT_INPUT_LOADED
    assert RESTORED_GLOBAL_STEP == session_initial_step
    assert session_context.restored_expected_step == session_initial_step
    assert REPLAY_BUFFER_RESTORED
    assert session_initial_step > 0
    assert session_final_step > session_initial_step
    assert session_summary.checkpoint_input_reference == CHECKPOINT_INPUT_REFERENCE
    assert MULTISESSION_CHECKPOINT_RESUME_PASS
else:
    raise AssertionError(f"Unsupported tracking mode: {session_context.tracking_mode}")
if mlflow_config.get("enabled", False):
    assert MLFLOW_TRACKING_PASS, "MLflow tracking validation did not pass."

bootstrap_diagnostics = inspect_experiment_state(
    BASE,
    session_context.project_run_id,
    config=session_config,
    tracking_uri=session_context.tracking_uri,
)
assert bootstrap_diagnostics.ok, bootstrap_diagnostics.as_dict()

print("HU009 training session status")
print("TRAINING_PROFILE=", profile_context.name)
print("FULL_TRAINING_READY=", FULL_TRAINING_READY)
print("full_training_gate_issues:", full_training_gate.issues)
print("runtime:", selected_runtime)
print("device:", selected_device)
print("tracking_mode:", session_context.tracking_mode)
print("project_run_id:", mlflow_metadata.project_run_id)
print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)
print("tracking_session_id:", mlflow_metadata.tracking_session_id)
print("checkpoint_input_reference:", CHECKPOINT_INPUT_REFERENCE)
print("checkpoint_input_loaded:", CHECKPOINT_INPUT_LOADED)
print("restored_global_step:", RESTORED_GLOBAL_STEP)
print("restored_expected_step:", session_context.restored_expected_step)
print("replay_buffer_restored:", REPLAY_BUFFER_RESTORED)
print("initial_global_step:", session_initial_step)
print("final_global_step:", session_final_step)
print("checkpoint_output_reference:", checkpoint_output_reference)
print("session_target_timesteps:", session_context.target_timesteps)
print("evaluation_episodes:", evaluation_summary.episodes)
print("evaluation_mean_reward:", evaluation_summary.mean_reward)
print("evaluation_epsilon:", evaluation_summary.epsilon)
print("mlflow_tracking_uri:", mlflow_metadata.tracking_uri)
print("mlflow_experiment:", mlflow_metadata.experiment_name)
print("manifest_path:", session_context.manifest_path)
print("manifest_updated:", EXPERIMENT_MANIFEST_UPDATED)
print("config_fingerprint:", session_context.config_fingerprint)
print("observation:", metadata.observation_shape, metadata.observation_dtype)
print("action_space:", metadata.action_space)
print("Preflight READY_FOR_TRAINING:", preflight_report.ready_for_training)
print("training_updates:", session_summary.training.updates_count)
print("training_initial_step:", session_summary.training.initial_global_step)
print("training_final_step:", session_summary.training.global_step)
print("checkpoint_path:", session_summary.checkpoint.path)
print("checkpoint_size_bytes:", session_summary.checkpoint.size_bytes)
print("replay_buffer_capacity:", profile_context.replay_buffer_memory.capacity)
print("replay_buffer_estimated_memory_gib:", round(profile_context.replay_buffer_memory.total_gib, 3))
print("runtime_ram_available_gib:", full_training_gate.ram_available_gib)
print("runtime_ram_margin_gib:", full_training_gate.ram_margin_gib)
print("session_artifacts:", MLFLOW_SESSION_ARTIFACTS)
print("effective_config_artifact:", effective_config_artifact)
print("SESSION_BOOTSTRAP_READY=True")
print("MULTISESSION_CHECKPOINT_RESUME_PASS=", MULTISESSION_CHECKPOINT_RESUME_PASS)
print("MLFLOW_TRACKING_PASS=", MLFLOW_TRACKING_PASS)


HU009 training session status
TRAINING_PROFILE= full
FULL_TRAINING_READY= True
full_training_gate_issues: []
runtime: Google Colab
device: cuda
tracking_mode: new
project_run_id: assault_ddqn_full_001
mlflow_run_id: 048a4acf1a3541b1bcfc88b278070912
tracking_session_id: session_001
checkpoint_input_reference: None
checkpoint_input_loaded: False
restored_global_step: None
restored_expected_step: None
replay_buffer_restored: False
initial_global_step: 0
final_global_step: 250000
checkpoint_output_reference: /content/drive/MyDrive/reinforcement_learning_reto_1/checkpoints/assault_ddqn_full_001/checkpoint_step_250000.pt
session_target_timesteps: 250000
evaluation_episodes: 10
evaluation_mean_reward: 569.1
evaluation_epsilon: 0.0
mlflow_tracking_uri: file:///content/drive/MyDrive/reinforcement_learning_reto_1/mlruns
mlflow_experiment: assault_ddqn
manifest_path: /content/drive/MyDrive/reinforcement_learning_reto_1/experiments/assault_ddqn_full_001/experiment_state.json
manifest_updated: True

In [ ]:
# 